# Calcualting aggregate statistic values

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

In [2]:
DATA_PATH = 'reports'
ALL_DATA = os.path.join(DATA_PATH, 'all.csv')

In [3]:
df = pd.read_csv(ALL_DATA)

## Aggregating results by condition

In [4]:
partition_by = ['self']

In [5]:
df0 = df[partition_by+['b']].groupby(by=partition_by).agg('mean').reset_index()
df0['std'] = df[partition_by+['b']].groupby(by=partition_by).agg('std').reset_index()['b']
df0['k'] = df[partition_by+['b']].groupby(by=partition_by).agg('count').reset_index()['b']
df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [6]:
df0['tstat'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['tstat', 'k']].values]

In [7]:
df0.head(n=50)

,self,b,std,k,se,tstat,p
0,False,-0.387600,0.943652,12019,0.008608,-45.030446,0.0
1,True,-0.486513,1.104611,10719,0.010669,-45.599785,0.0


In [8]:
df0.to_csv(
    os.path.join(DATA_PATH, 'study-level-statistics.csv'),
    index=False,
    encoding='utf-8'
)

## T-statistics

### Difference from zero

In [7]:
null = 0
tstat = ((df0['b'] - null) / df0['se']).values
list(zip(df0[partition_by].values, tstat, t.sf(tstat.__abs__(), df=df0['k'].values - 1)))

[(array([False]), np.float64(-45.030446477766944), np.float64(0.0)),
 (array([ True]), np.float64(-45.599784805205246), np.float64(0.0))]

### Difference from forest fire model

In [8]:
null = -.5
tstat = ((df0['b'] - null) / df0['se']).values
list(zip(df0[partition_by].values, tstat, t.sf(tstat.__abs__(), df=df0['k'].values - 1)))

[(array([False]),
  np.float64(13.058352413405846),
  np.float64(5.220638826974203e-39)),
 (array([ True]),
  np.float64(1.2640586181091986),
  np.float64(0.10311823888548585))]

## Visualizations!

In [9]:
import plotly.figure_factory as ff

In [14]:
hist_data = [
    df['b'].loc[~df['self']].values,
    df['b'].loc[df['self']].values,
]

groups = [
    'other',
    'self',
]

colors = [
    '#0504aa',
    '#FFC300',
]

In [15]:
fig = ff.create_distplot(hist_data,groups,show_hist=False,colors=colors)